# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [3]:
# Load the dataset 
df = pd.read_csv('data/AviationData.csv', encoding='latin-1')

# Inspect basic information and datatypes
print(df.info())

# Inspect summary statistics for numerical columns
display(df.describe())

# Check for missing values across the dataset
print(df.isna().sum())

C:\Users\Administrator\AppData\Local\Temp\ipykernel_11728\3834597282.py:2: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/AviationData.csv', encoding='latin-1')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


Event.Id                      0
Investigation.Type            0
Accident.Number               0
Event.Date                    0
Location                     52
Country                     226
Latitude                  54507
Longitude                 54516
Airport.Code              38757
Airport.Name              36185
Injury.Severity            1000
Aircraft.damage            3194
Aircraft.Category         56602
Registration.Number        1382
Make                         63
Model                        92
Amateur.Built               102
Number.of.Engines          6084
Engine.Type                7096
FAR.Description           56866
Schedule                  76307
Purpose.of.flight          6192
Air.carrier               72241
Total.Fatal.Injuries      11401
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Uninjured            5912
Weather.Condition          4492
Broad.phase.of.flight     27165
Report.Status              6384
Publication.Date          13771
dtype: i

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [ ]:
# Load the dataset (assuming standard NTSB format, latin-1 encoding handles special characters in this common dataset)
df = pd.read_csv('AviationData.csv', encoding='latin-1')

# Inspect basic information and datatypes
print(df.info())

# Inspect summary statistics for numerical columns
display(df.describe())

# Check for missing values across the dataset
print(df.isna().sum())

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [4]:
# Impute missing injury/passenger counts with 0 
injury_columns = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']
df[injury_columns] = df[injury_columns].fillna(0)

# Estimate total passengers on board
df['Total_Passengers'] = df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries'] + df['Total.Minor.Injuries'] + df['Total.Uninjured']

# Calculate the fraction of passengers with fatal or serious injuries
# We use np.where to avoid division by zero errors for flights with 0 recorded total passengers
df['Severe_Injury_Rate'] = np.where(
    df['Total_Passengers'] > 0, 
    (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['Total_Passengers'], 
    np.nan
)

# Drop rows where passenger count was 0 or unresolvable, as they are not useful for a passenger safety metric
df = df.dropna(subset=['Severe_Injury_Rate']).copy()

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [7]:
# Inspect current values
print(df['Aircraft.damage'].value_counts(dropna=False))

# Impute missing values with 'Unknown'
df['Aircraft.damage'] = df['Aircraft.damage'].fillna('Unknown')

# Create binary derived column: 1 if Destroyed, 0 otherwise
df['Is_Destroyed'] = df['Aircraft.damage'].apply(lambda x: 1 if str(x).upper() == 'DESTROYED' else 0)

Aircraft.damage
Substantial    63879
Destroyed      18527
NaN             2590
Minor           2492
Unknown           92
Name: count, dtype: int64


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

**Cleaning Tasks Identified for 'Make':**
1. Standardize string casing (convert all to uppercase).
2. Strip leading and trailing whitespace.
3. Merge common inconsistencies (e.g., 'MCDONNELL DOUGLAS' and 'MCDONNELL-DOUGLAS').
4. Filter out makes with fewer than 50 total records to ensure statistical robustness.

In [8]:
# Standardize casing and spacing
df['Make'] = df['Make'].astype(str).str.upper().str.strip()

# Handle common aliases/inconsistencies (Add to this dictionary based on your specific dataset's unique errors)
make_replacements = {
    'MCDONNELL DOUGLAS': 'MCDONNELL-DOUGLAS',
    'AIRBUS INDUSTRIE': 'AIRBUS',
    'DEHAVILLAND': 'DE HAVILLAND'
}
df['Make'] = df['Make'].replace(make_replacements)

# Filter for Makes with >= 50 occurrences
make_counts = df['Make'].value_counts()
robust_makes = make_counts[make_counts >= 50].index
df = df[df['Make'].isin(robust_makes)].copy()

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [9]:
# Drop rows with NaN in the Model column
df = df.dropna(subset=['Model']).copy()

# Standardize Model names
df['Model'] = df['Model'].astype(str).str.upper().str.strip()

# Create a unique identifier combining Make and Model (Models like '172' exist across different makes)
df['Make_Model'] = df['Make'] + "_" + df['Model']

print(f"Number of unique Make_Model configurations: {df['Make_Model'].nunique()}")

Number of unique Make_Model configurations: 6786


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [10]:
cols_to_clean = ['Engine.Type', 'Weather.Condition', 'Broad.phase.of.flight', 'Purpose.of.flight']

for col in cols_to_clean:
    if col in df.columns:
        # Standardize strings, treat "UNK" or "UNKNOWN" as NaN for cleaner analysis
        df[col] = df[col].astype(str).str.upper().str.strip()
        df[col] = df[col].replace(['NAN', 'UNK', 'UNKNOWN'], np.nan)

# Clean Number of Engines (convert to numeric, coerce errors to NaN)
if 'Number.of.Engines' in df.columns:
    df['Number.of.Engines'] = pd.to_numeric(df['Number.of.Engines'], errors='coerce')

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [11]:
# Calculate percentage of missing values per column
missing_pct = df.isna().mean()

# Drop columns that are missing more than 60% of their data, as they are too sparse for reliable analysis
cols_to_drop = missing_pct[missing_pct > 0.6].index
df_cleaned = df.drop(columns=cols_to_drop)

# Drop irrelevant columns for this specific analysis to save memory (e.g., flight numbers, registration)
irrelevant_cols = ['Registration.Number', 'Schedule', 'FAR.Description']
df_cleaned = df_cleaned.drop(columns=[col for col in irrelevant_cols if col in df_cleaned.columns])

print("Final columns:", df_cleaned.columns.tolist())

Final columns: ['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date', 'Location', 'Country', 'Airport.Code', 'Airport.Name', 'Injury.Severity', 'Aircraft.damage', 'Make', 'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'Purpose.of.flight', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status', 'Publication.Date', 'Total_Passengers', 'Severe_Injury_Rate', 'Is_Destroyed', 'Make_Model']


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [12]:
df_cleaned.to_csv('data/cleaned_aviation_data.csv', index=False)
print("Data successfully saved to 'cleaned_aviation_data.csv'")

Data successfully saved to 'cleaned_aviation_data.csv'
